# 08 — Quando uma única memória reflexiva ajuda?

Este notebook estima o efeito **pareado** de acrescentar exatamente **uma
memória** ao prompt de um modelo pequeno. Para cada questão de validação, ele
compara a mesma resposta sem memória com respostas que recebem memórias vindas
de diferentes similaridades.

O desenho inclui:

1. deduplicação de treino/validação por `context + question`;
2. embeddings com a pergunta antes do contexto e auditoria de truncamento;
3. uma escada de três similaridades-alvo, o vizinho top-1 e uma memória
   placebo de baixa similaridade;
4. reflexões `simple` e `complex`, ambas answer-aware, com formato curto para
   modelos abaixo de 8B;
5. uma única memória em cada prompt experimental;
6. correção pela letra final, usando o LLM juiz somente quando o formato falha;
7. split fixo de calibração/teste;
8. curva de ganho pareado com bootstrap agrupado por questão;
9. threshold descoberto na calibração e política avaliada apenas no teste;
10. resultados separados por modelo, dataset, profundidade e origem da
    memória (`all`, `errors`, `correct`).

O notebook **não filtra nem regenera reflexões que revelem conteúdo da
resposta**. O cumprimento das instruções faz parte do comportamento medido.


In [ ]:
import os
import sys
from pathlib import Path

# Precisa ser definido antes de importar rmcq/torch/vLLM.
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("RMCQ_NOTEBOOK_GPU", "3")

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "rmcq").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Não encontrei a raiz do repositório contendo rmcq/")

import rmcq

print(rmcq.env_summary())


In [ ]:
import gc
import hashlib
import json
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rmcq import ROOT
from rmcq.backends import get_backend
from rmcq.backends.base import GenParams
from rmcq.thresholds import (
    calibration_or_test,
    clustered_bootstrap_curve,
    extract_final_answer,
    normalize_stem,
    paired_bootstrap_difference,
    select_similarity_ladder,
    sustained_threshold,
    threshold_bootstrap_ci,
)

DATA_DIR = ROOT / "data" / "processed"
RESULTS_ROOT = ROOT / "data" / "results" / "similarity_threshold_v2"

DATASETS = ["aqua", "arc", "gsm8k", "logiqa2", "openbookqa"]

# Todos têm estritamente menos de 8B parâmetros. Para um ensaio rápido, deixe
# somente um; o threshold final deve ser replicado nos dois.
STUDENT_MODELS = ["phi4-mini", "mistral-7b"]

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SEED = 42
N_VALIDATION_PER_DATASET = 1565
GSM8K_VALIDATION_FALLBACK_SIZE = 1000
CALIBRATION_FRACTION = 0.60

# Uma memória perto de cada alvo, mais top-1 e placebo. Cada resposta recebe
# somente UMA delas. Os valores realizados, e não os alvos, entram na análise.
SIMILARITY_TARGETS = [0.30, 0.50, 0.70]
PLACEBO_BOTTOM_QUANTILE = 0.20
REFLECTION_DEPTHS = ["simple", "complex"]
SOURCE_POOLS = ["all", "errors", "correct"]

ANSWER_MAX_NEW_TOKENS = 400
REFLECTION_MAX_NEW_TOKENS = 180
JUDGE_MAX_NEW_TOKENS = 80
GENERATION_BATCH_SIZE = 512
RESUME = True
# Faz parte do hash de cada geração. Mude quando a renderização/tokenização do
# backend mudar, para nunca misturar respostas produzidas por pipelines distintos.
GENERATION_CACHE_VERSION = "direct-chat-template-token-ids-v2"

KERNEL_BANDWIDTH = 0.08
N_BOOTSTRAP_CURVE = 1000
N_BOOTSTRAP_POLICY = 2000
MIN_EFFECTIVE_N = 20
SUSTAINED_GRID_POINTS = 5

print("datasets:", DATASETS)
print("modelos <8B:", STUDENT_MODELS)
print("similaridades-alvo:", SIMILARITY_TARGETS, "+ top-1 + placebo")
print("uma memória por prompt experimental")


## Prompts versionados

As reflexões recebem a resposta correta como feedback privado. Isso deve ser
interpretado como **oracle-guided reflection**, não como reflexão baseada
somente em `CORRECT/INCORRECT`. O texto transferido é curto e estruturado,
mas não há pós-filtro de vazamento.

O prompt experimental diz explicitamente que a memória veio de outro problema
e que pode ser ignorada. Ele contém exatamente um bloco `Memory:`.


In [ ]:
ANSWER_PROMPT = """You are answering a multiple-choice question.

Question: {question}

Options:
{options}

Instructions:
- Reason carefully before choosing.
- Choose exactly one option.
- End with this exact line and nothing after it:
FINAL ANSWER: <letter>"""

SIMPLE_REFLECTION_PROMPT = """You are reviewing your answer to a completed multiple-choice problem.
You are given the problem, your previous response, whether it was correct, and the correct answer as private feedback.

Write exactly four lines and no more than 100 words total:
Approach: State the reasoning method you used.
Diagnosis: Identify the specific reasoning step that succeeded or failed.
Transfer rule: Give one general rule in the form "If ..., then ..." for similar problems.
Applicability: State when that rule should not be used.

Do not solve the problem again. Do not state or quote the correct answer, an option label, or an option text.
Focus on transferable reasoning. Do not give vague advice such as "be careful" or "think harder"."""

COMPLEX_REFLECTION_PROMPT = """You are reviewing your answer to a completed multiple-choice problem.
You are given the problem, your previous response, whether it was correct, and the correct answer as private feedback.

Write exactly six lines and no more than 140 words total:
Task type: Identify the general kind of reasoning required.
Strategy: Describe the reasoning strategy you used.
Evidence check: State the key evidence, relationship, or constraint that you used or missed.
Failure test: Give one check that would have exposed the mistake or confirmed the reasoning.
Transfer rule: Give a precise rule in the form "If ..., then ..." for future problems.
Boundary: State when that rule would not apply.

Do not solve the problem again. Do not state or quote the correct answer, an option label, or an option text.
Focus on transferable reasoning rather than source-specific conclusions. Do not give vague advice."""

REFLECTION_PROMPTS = {
    "simple": SIMPLE_REFLECTION_PROMPT,
    "complex": COMPLEX_REFLECTION_PROMPT,
}

MEMORY_PROMPT = """The following is one memory written after solving a DIFFERENT multiple-choice problem.
Use it only if its reasoning rule applies. Do not copy its conclusion or assume that its option labels match this problem.
Solve the current problem independently, then use the memory as a check.

Memory:
{memory}

Question: {question}

Options:
{options}

Instructions:
- Reason carefully before choosing.
- Choose exactly one option.
- End with this exact line and nothing after it:
FINAL ANSWER: <letter>"""

JUDGE_PROMPT = """You are grading a multiple-choice response.

Question: {question}

Options:
{options}

Correct option: {correct_label}) {correct_text}

Candidate response:
{response}

Decide only which option the candidate ultimately selected.
End with this exact line and nothing after it:
Verdict: <CORRECT or INCORRECT>"""


def format_options(choices):
    return "\n".join(f"{c['label']}) {c['text']}" for c in choices)


def format_question(item):
    context = (item.get("context") or "").strip()
    question = item["question"].strip()
    return f"{context}\n\n{question}" if context else question


def build_answer_prompt(item):
    return ANSWER_PROMPT.format(
        question=format_question(item), options=format_options(item["choices"])
    )


def build_reflection_prompt(item, previous_answer, was_correct, depth):
    correct_label = item["answerKey"]
    correct_text = next(c["text"] for c in item["choices"] if c["label"] == correct_label)
    outcome = "CORRECT" if was_correct else "INCORRECT"
    return (
        f"{REFLECTION_PROMPTS[depth]}\n\n"
        f"Problem:\n{format_question(item)}\n\n"
        f"Options:\n{format_options(item['choices'])}\n\n"
        f"Your previous response:\n{previous_answer.strip()}\n\n"
        f"Outcome: {outcome}\n"
        f"Correct answer for private feedback: {correct_label}) {correct_text}"
    )


def build_memory_prompt(item, memory):
    assert MEMORY_PROMPT.count("Memory:") == 1
    prompt = MEMORY_PROMPT.format(
        memory=(memory or "").strip(),
        question=format_question(item),
        options=format_options(item["choices"]),
    )
    return prompt


def build_judge_prompt(item, response):
    label = item["answerKey"]
    text = next(c["text"] for c in item["choices"] if c["label"] == label)
    return JUDGE_PROMPT.format(
        question=format_question(item), options=format_options(item["choices"]),
        correct_label=label, correct_text=text, response=(response or "").strip(),
    )


def parse_judge(text):
    import re
    match = re.search(r"Verdict:\s*(CORRECT|INCORRECT)", text or "", re.I)
    return None if not match else match.group(1).upper() == "CORRECT"


signature_payload = {
    "pipeline_version": "single-memory-multisim-v2.1",
    "generation_cache_version": GENERATION_CACHE_VERSION,
    "models": STUDENT_MODELS,
    "datasets": DATASETS,
    "validation_cap": N_VALIDATION_PER_DATASET,
    "gsm8k_holdout_size": GSM8K_VALIDATION_FALLBACK_SIZE,
    "embedding": EMBEDDING_MODEL_NAME,
    "targets": SIMILARITY_TARGETS,
    "placebo_bottom_quantile": PLACEBO_BOTTOM_QUANTILE,
    "calibration_fraction": CALIBRATION_FRACTION,
    "depths": REFLECTION_DEPTHS,
    "answer_prompt": ANSWER_PROMPT,
    "reflection_prompts": REFLECTION_PROMPTS,
    "memory_prompt": MEMORY_PROMPT,
    "kernel_bandwidth": KERNEL_BANDWIDTH,
    "bootstrap_curve": N_BOOTSTRAP_CURVE,
    "bootstrap_policy": N_BOOTSTRAP_POLICY,
    "min_effective_n": MIN_EFFECTIVE_N,
    "sustained_grid_points": SUSTAINED_GRID_POINTS,
    "seed": SEED,
}
EXPERIMENT_ID = hashlib.sha256(
    json.dumps(signature_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:12]
OUT_DIR = RESULTS_ROOT / EXPERIMENT_ID
for sub in ("retrieval", "generations", "analysis", "plots"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(signature_payload | {"experiment_id": EXPERIMENT_ID}, f, indent=2)
print("experiment_id:", EXPERIMENT_ID)
print("saída:", OUT_DIR.relative_to(ROOT))


## Dados: holdout, deduplicação e split de análise

O stem normalizado é `context + question`. Primeiro removemos duplicatas
internas; depois removemos do treino qualquer stem presente na validação. Isso
elimina a contaminação observada nos splits originais. A divisão
calibração/teste é estável por `uid` e compartilhada por modelos e condições.


In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def save_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    with open(temp, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    temp.replace(path)


def sample_items(items, n, seed):
    if n is None or len(items) <= n:
        return list(items)
    return random.Random(seed).sample(items, n)


def dedupe_by_stem(items):
    seen = set()
    kept = []
    for item in items:
        key = normalize_stem(item)
        if key not in seen:
            seen.add(key)
            kept.append(item)
    return kept, len(items) - len(kept)


def load_dataset_split(dataset):
    train = load_jsonl(DATA_DIR / dataset / "train.jsonl")
    val_path = DATA_DIR / dataset / "validation.jsonl"
    source = "validation.jsonl"
    if val_path.exists():
        val = load_jsonl(val_path)
    else:
        indices = list(range(len(train)))
        random.Random(SEED).shuffle(indices)
        held_out = set(indices[:GSM8K_VALIDATION_FALLBACK_SIZE])
        val = [train[i] for i in sorted(held_out)]
        train = [item for i, item in enumerate(train) if i not in held_out]
        source = "holdout determinístico do train"

    val = sample_items(val, N_VALIDATION_PER_DATASET, SEED)
    val, val_internal_duplicates = dedupe_by_stem(val)
    train, train_internal_duplicates = dedupe_by_stem(train)
    val_stems = {normalize_stem(item) for item in val}
    before_cross = len(train)
    train = [item for item in train if normalize_stem(item) not in val_stems]
    cross_split_removed = before_cross - len(train)

    audit = {
        "dataset": dataset,
        "source": source,
        "train": len(train),
        "validation": len(val),
        "train_internal_duplicates_removed": train_internal_duplicates,
        "validation_internal_duplicates_removed": val_internal_duplicates,
        "cross_split_stems_removed_from_train": cross_split_removed,
    }
    return train, val, audit


dataset_state = {}
audits = []
for dataset in DATASETS:
    train, val, audit = load_dataset_split(dataset)
    for item in val:
        item["analysis_split"] = calibration_or_test(
            f"{dataset}:{item['uid']}", CALIBRATION_FRACTION, SEED
        )
    dataset_state[dataset] = {
        "train": train,
        "val": val,
        "train_by_uid": {x["uid"]: x for x in train},
        "val_by_uid": {x["uid"]: x for x in val},
    }
    audits.append(audit)

audit_df = pd.DataFrame(audits)
audit_df.to_csv(OUT_DIR / "retrieval" / "dedup_audit.csv", index=False)
display(audit_df)
for dataset, state in dataset_state.items():
    print(dataset, Counter(x["analysis_split"] for x in state["val"]))


## Embeddings e escada de similaridade

A representação começa pela pergunta, seguida de contexto e alternativas sem
letras. Isso protege a pergunta contra truncamento e torna questões curtas de
OpenBookQA mais informativas. A taxa de textos acima do limite do encoder é
salva para auditoria.

Para cada alvo são escolhidas memórias próximas de 0.30, 0.50 e 0.70, uma
memória top-1 e uma placebo sorteada deterministicamente entre os 20% menos
similares. O score efetivamente obtido é usado na estatística.


In [ ]:
from sentence_transformers import SentenceTransformer


def embedding_text(item):
    parts = [f"Question: {item['question'].strip()}"]
    context = (item.get("context") or "").strip()
    if context:
        parts.append(f"Context: {context}")
    parts.append("Options: " + " | ".join(c["text"].strip() for c in item["choices"]))
    return "\n".join(parts)


def token_length_audit(tokenizer, texts, max_length, batch_size=2048):
    lengths = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(
            texts[start:start + batch_size], truncation=False,
            add_special_tokens=True, return_length=True,
        )
        lengths.extend(batch["length"])
    arr = np.asarray(lengths)
    return {
        "n": len(arr), "max_seq_length": int(max_length),
        "n_over_limit": int((arr > max_length).sum()),
        "rate_over_limit": float((arr > max_length).mean()),
        "p95_tokens": float(np.percentile(arr, 95)),
        "max_tokens": int(arr.max()),
    }


embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
retrieval_rows = []
truncation_rows = []

for dataset, state in dataset_state.items():
    train_texts = [embedding_text(x) for x in state["train"]]
    val_texts = [embedding_text(x) for x in state["val"]]
    max_len = int(embedder.max_seq_length)
    for split, texts in (("train", train_texts), ("validation", val_texts)):
        truncation_rows.append(
            {"dataset": dataset, "split": split}
            | token_length_audit(embedder.tokenizer, texts, max_len)
        )

    train_emb = embedder.encode(
        train_texts, batch_size=256, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    )
    val_emb = embedder.encode(
        val_texts, batch_size=256, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    )
    similarities = val_emb @ train_emb.T
    source_ids = [x["uid"] for x in state["train"]]

    dataset_pairs = []
    for i, item in enumerate(state["val"]):
        selected = select_similarity_ladder(
            similarities[i], source_ids, SIMILARITY_TARGETS,
            seed_key=f"{SEED}:{dataset}:{item['uid']}",
            placebo_quantile=PLACEBO_BOTTOM_QUANTILE,
        )
        for pair in selected:
            row = {
                "dataset": dataset,
                "val_uid": item["uid"],
                "analysis_split": item["analysis_split"],
            } | pair
            dataset_pairs.append(row)
            retrieval_rows.append(row)
    state["pairs"] = dataset_pairs
    save_jsonl(OUT_DIR / "retrieval" / f"{dataset}.jsonl", dataset_pairs)

    del train_emb, val_emb, similarities, train_texts, val_texts
    gc.collect()

truncation_df = pd.DataFrame(truncation_rows)
truncation_df.to_csv(OUT_DIR / "retrieval" / "embedding_truncation.csv", index=False)
retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_df.to_csv(OUT_DIR / "retrieval" / "pairs.csv", index=False)
display(truncation_df)
display(
    retrieval_df.groupby(["dataset", "arm", "level"])["similarity"]
    .agg(["count", "min", "median", "max"]).reset_index()
)

# SentenceTransformer permanece na GPU enquanto o objeto existir.
del embedder
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass


## Geração retomável e avaliação

Cada etapa grava checkpoints identificados pelo hash do prompt. Uma execução
interrompida continua do último lote completo. O parser da linha
`FINAL ANSWER:` é a métrica primária; somente respostas sem essa linha passam
por um juiz fallback.

As reflexões são geradas uma vez por fonte e profundidade e reutilizadas nas
questões que recuperam aquela fonte. Cada prompt de avaliação contém apenas
uma delas.


In [ ]:
def cache_key(*parts):
    return json.dumps(parts, ensure_ascii=False, separators=(",", ":"))


def prompt_hash(prompt):
    payload = f"{GENERATION_CACHE_VERSION}\0{prompt}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def load_cache(path):
    if not (RESUME and path.exists()):
        return {}
    rows = load_jsonl(path)
    return {row["key"]: row for row in rows}


def cached_generate(backend, path, keyed_prompts, max_new_tokens, desc):
    cache = load_cache(path)
    missing = [
        (key, prompt) for key, prompt in keyed_prompts.items()
        if key not in cache or cache[key].get("prompt_hash") != prompt_hash(prompt)
    ]
    print(f"{desc}: total={len(keyed_prompts)} cache={len(keyed_prompts)-len(missing)} faltando={len(missing)}")
    for start in range(0, len(missing), GENERATION_BATCH_SIZE):
        batch = missing[start:start + GENERATION_BATCH_SIZE]
        generations = backend.generate(
            [prompt for _, prompt in batch],
            GenParams(max_new_tokens=max_new_tokens),
            desc=f"{desc} [{start + 1}-{start + len(batch)}]",
        )
        for (key, prompt), generation in zip(batch, generations):
            cache[key] = {
                "key": key,
                "prompt_hash": prompt_hash(prompt),
                "text": generation.text,
                "prompt_tokens": generation.prompt_tokens,
                "completion_tokens": generation.completion_tokens,
                "finish_reason": generation.finish_reason,
            }
        save_jsonl(path, cache.values())
    return {key: cache[key]["text"] for key in keyed_prompts}


def resolve_correctness(backend, model_dir, stage, texts, items):
    correctness = {}
    method = {}
    fallback_prompts = {}
    for key, text in texts.items():
        item = items[key]
        answer = extract_final_answer(text)
        if answer is not None:
            correctness[key] = answer == item["answerKey"]
            method[key] = "parser"
        else:
            fallback_prompts[key] = build_judge_prompt(item, text)

    if fallback_prompts:
        judged = cached_generate(
            backend, model_dir / f"judge_fallback_{stage}.jsonl",
            fallback_prompts, JUDGE_MAX_NEW_TOKENS, f"juiz fallback: {stage}",
        )
        for key, text in judged.items():
            correctness[key] = parse_judge(text)
            method[key] = "judge_fallback" if correctness[key] is not None else "unresolved"
    return correctness, method


In [ ]:
all_result_rows = []

for model in STUDENT_MODELS:
    print("\n" + "=" * 90)
    print("MODELO", model)
    model_dir = OUT_DIR / "generations" / model
    model_dir.mkdir(parents=True, exist_ok=True)

    selected_sources = {
        (row["dataset"], row["source_uid"])
        for row in retrieval_rows
    }
    source_items = {
        cache_key("source", dataset, uid): dataset_state[dataset]["train_by_uid"][uid]
        for dataset, uid in sorted(selected_sources)
    }
    val_items = {
        cache_key("baseline", dataset, item["uid"]): item
        for dataset, state in dataset_state.items() for item in state["val"]
    }

    with get_backend(model, kind="vllm") as backend:
        source_prompts = {key: build_answer_prompt(item) for key, item in source_items.items()}
        source_answers = cached_generate(
            backend, model_dir / "source_answers.jsonl", source_prompts,
            ANSWER_MAX_NEW_TOKENS, "respostas das fontes",
        )
        source_correct, source_method = resolve_correctness(
            backend, model_dir, "sources", source_answers, source_items
        )

        reflections = {}
        for depth in REFLECTION_DEPTHS:
            prompts = {
                cache_key("reflection", depth, *json.loads(key)[1:]):
                build_reflection_prompt(item, source_answers[key], source_correct[key], depth)
                for key, item in source_items.items()
                if source_correct.get(key) is not None
            }
            outputs = cached_generate(
                backend, model_dir / f"reflections_{depth}.jsonl", prompts,
                REFLECTION_MAX_NEW_TOKENS, f"reflexões {depth}",
            )
            for key, text in outputs.items():
                _, _, dataset, uid = json.loads(key)
                reflections[(dataset, uid, depth)] = text

        baseline_prompts = {key: build_answer_prompt(item) for key, item in val_items.items()}
        baseline_answers = cached_generate(
            backend, model_dir / "validation_baseline.jsonl", baseline_prompts,
            ANSWER_MAX_NEW_TOKENS, "baseline de validação",
        )
        baseline_correct, baseline_method = resolve_correctness(
            backend, model_dir, "baseline", baseline_answers, val_items
        )

        for depth in REFLECTION_DEPTHS:
            condition_prompts = {}
            condition_items = {}
            pair_by_key = {}
            for pair in retrieval_rows:
                memory = reflections.get((pair["dataset"], pair["source_uid"], depth))
                if memory is None:
                    continue
                item = dataset_state[pair["dataset"]]["val_by_uid"][pair["val_uid"]]
                key = cache_key(
                    "memory", pair["dataset"], pair["val_uid"], depth,
                    pair["arm"], pair["level"], pair["source_uid"],
                )
                condition_prompts[key] = build_memory_prompt(item, memory)
                condition_items[key] = item
                pair_by_key[key] = pair

            condition_answers = cached_generate(
                backend, model_dir / f"validation_with_memory_{depth}.jsonl",
                condition_prompts, ANSWER_MAX_NEW_TOKENS,
                f"validação com uma memória {depth}",
            )
            condition_correct, condition_method = resolve_correctness(
                backend, model_dir, f"memory_{depth}", condition_answers, condition_items
            )

            for key, pair in pair_by_key.items():
                baseline_key = cache_key("baseline", pair["dataset"], pair["val_uid"])
                source_key = cache_key("source", pair["dataset"], pair["source_uid"])
                b = baseline_correct.get(baseline_key)
                m = condition_correct.get(key)
                s = source_correct.get(source_key)
                all_result_rows.append({
                    "model": model,
                    "dataset": pair["dataset"],
                    "val_uid": pair["val_uid"],
                    "source_uid": pair["source_uid"],
                    "analysis_split": pair["analysis_split"],
                    "depth": depth,
                    "arm": pair["arm"],
                    "level": pair["level"],
                    "requested_similarity": pair["requested_similarity"],
                    "similarity": pair["similarity"],
                    "source_correct": s,
                    "baseline_correct": b,
                    "memory_correct": m,
                    "delta": None if b is None or m is None else int(m) - int(b),
                    "baseline_eval_method": baseline_method.get(baseline_key),
                    "memory_eval_method": condition_method.get(key),
                    "source_eval_method": source_method.get(source_key),
                })

    model_rows = [row for row in all_result_rows if row["model"] == model]
    save_jsonl(model_dir / "outcomes.jsonl", model_rows)
    pd.DataFrame(model_rows).to_csv(model_dir / "outcomes.csv", index=False)
    print(model, "linhas de resultado:", len(model_rows))

results_df = pd.DataFrame(all_result_rows)
results_df.to_csv(OUT_DIR / "analysis" / "all_outcomes.csv", index=False)
print("total:", len(results_df), "->", (OUT_DIR / "analysis" / "all_outcomes.csv").relative_to(ROOT))
display(results_df.head())


## Estimativa do threshold

Na calibração, estimamos

`E[correct_with_memory - correct_baseline | similarity]`

por kernel gaussiano. O bootstrap reamostra questões de validação inteiras,
mantendo juntas suas várias similaridades. O threshold pontual é o primeiro
ponto com ganho não negativo por cinco pontos consecutivos e suporte efetivo
mínimo. Também reportamos:

- intervalo bootstrap do threshold;
- frequência com que o bootstrap conseguiu identificar um threshold;
- primeiro ponto com evidência positiva (`CI low > 0`);
- resultados separados para todas as fontes, fontes erradas e fontes corretas.

O threshold é aplicado somente ao vizinho `top` no split de teste. Assim, o
teste mede uma política executável: usar uma única memória top-1 apenas quando
sua similaridade supera o corte aprendido.


In [ ]:
def pool_mask(df, pool):
    if pool == "errors":
        return df["source_correct"] == False
    if pool == "correct":
        return df["source_correct"] == True
    return df["source_correct"].notna()


curve_rows = []
threshold_rows = []

for (model, dataset, depth), group in results_df.groupby(["model", "dataset", "depth"]):
    calibration = group[
        (group["analysis_split"] == "calibration")
        & (group["arm"] == "retrieved")
        & group["delta"].notna()
    ].copy()
    for pool in SOURCE_POOLS:
        sub = calibration[pool_mask(calibration, pool)].copy()
        n_clusters = sub["val_uid"].nunique()
        if len(sub) < 40 or n_clusters < 20 or sub["similarity"].nunique() < 10:
            threshold_rows.append({
                "model": model, "dataset": dataset, "depth": depth, "pool": pool,
                "n_rows_calibration": len(sub), "n_questions_calibration": n_clusters,
                "threshold": None, "threshold_ci_low": None, "threshold_ci_high": None,
                "threshold_identification_rate": 0.0, "confident_help_threshold": None,
                "confident_harm_region_start": None,
                "status": "insufficient_data",
            })
            continue

        lo, hi = np.quantile(sub["similarity"], [0.02, 0.98])
        grid = np.linspace(lo, hi, 81)
        records = sub[["val_uid", "similarity", "delta"]].to_dict("records")
        curve = clustered_bootstrap_curve(
            records, grid, bandwidth=KERNEL_BANDWIDTH,
            n_boot=N_BOOTSTRAP_CURVE,
            seed=SEED + sum(ord(c) for c in f"{model}{dataset}{depth}{pool}"),
        )
        threshold = sustained_threshold(
            grid, curve["estimate"], curve["effective_n"],
            min_effective_n=MIN_EFFECTIVE_N, consecutive=SUSTAINED_GRID_POINTS,
        )
        ci_low, ci_high, identification_rate = threshold_bootstrap_ci(
            grid, curve["bootstrap"], curve["effective_n"],
            min_effective_n=MIN_EFFECTIVE_N, consecutive=SUSTAINED_GRID_POINTS,
        )
        confident_help = sustained_threshold(
            grid, curve["low"], curve["effective_n"],
            min_effective_n=MIN_EFFECTIVE_N, consecutive=SUSTAINED_GRID_POINTS,
        )
        # A região de dano exige que até o limite superior do IC esteja abaixo
        # de zero por uma sequência sustentada de pontos.
        confident_harm = sustained_threshold(
            grid, -curve["high"], curve["effective_n"],
            min_effective_n=MIN_EFFECTIVE_N, consecutive=SUSTAINED_GRID_POINTS,
        )
        status = "identified" if threshold is not None else "no_crossing"
        threshold_rows.append({
            "model": model, "dataset": dataset, "depth": depth, "pool": pool,
            "n_rows_calibration": len(sub), "n_questions_calibration": n_clusters,
            "threshold": threshold, "threshold_ci_low": ci_low,
            "threshold_ci_high": ci_high,
            "threshold_identification_rate": identification_rate,
            "confident_help_threshold": confident_help,
            "confident_harm_region_start": confident_harm,
            "status": status,
        })
        for i, similarity in enumerate(grid):
            curve_rows.append({
                "model": model, "dataset": dataset, "depth": depth, "pool": pool,
                "similarity": similarity, "effect": curve["estimate"][i],
                "ci_low": curve["low"][i], "ci_high": curve["high"][i],
                "effective_n": curve["effective_n"][i],
            })

threshold_df = pd.DataFrame(threshold_rows)
curves_df = pd.DataFrame(curve_rows)
threshold_df.to_csv(OUT_DIR / "analysis" / "thresholds_calibration.csv", index=False)
curves_df.to_csv(OUT_DIR / "analysis" / "effect_curves_calibration.csv", index=False)
display(threshold_df)


In [ ]:
from scipy.stats import binomtest


def add_mcnemar(result):
    discordant = result["helped"] + result["harmed"]
    result["mcnemar_p"] = (
        float(binomtest(result["helped"], discordant, 0.5).pvalue)
        if discordant else 1.0
    )
    return result


policy_rows = []
for threshold_row in threshold_rows:
    model = threshold_row["model"]
    dataset = threshold_row["dataset"]
    depth = threshold_row["depth"]
    pool = threshold_row["pool"]
    threshold = threshold_row["threshold"]
    group = results_df[
        (results_df["model"] == model)
        & (results_df["dataset"] == dataset)
        & (results_df["depth"] == depth)
        & (results_df["analysis_split"] == "test")
    ].copy()
    top = group[(group["arm"] == "retrieved") & (group["level"] == "top")].copy()
    top = top[top["baseline_correct"].notna() & top["memory_correct"].notna()]
    if top.empty:
        continue
    eligible = pool_mask(top, pool)

    # Always-top respeita o pool, mas não aplica threshold.
    always = np.where(eligible, top["memory_correct"], top["baseline_correct"]).astype(float)
    always_result = add_mcnemar(paired_bootstrap_difference(
        top["baseline_correct"].astype(float), always,
        n_boot=N_BOOTSTRAP_POLICY, seed=SEED,
    ))
    policy_rows.append({
        "model": model, "dataset": dataset, "depth": depth, "pool": pool,
        "policy": "always_top", "threshold": None,
    } | always_result)

    if threshold is not None:
        use_memory = eligible & (top["similarity"] >= threshold)
        policy = np.where(use_memory, top["memory_correct"], top["baseline_correct"]).astype(float)
        policy_result = add_mcnemar(paired_bootstrap_difference(
            top["baseline_correct"].astype(float), policy,
            n_boot=N_BOOTSTRAP_POLICY, seed=SEED + 1,
        ))
        policy_rows.append({
            "model": model, "dataset": dataset, "depth": depth, "pool": pool,
            "policy": "threshold_top", "threshold": threshold,
            "memory_use_rate": float(use_memory.mean()),
        } | policy_result)

    # Placebo é um controle geral e aparece uma vez por depth/dataset/model.
    if pool == "all":
        placebo = group[group["arm"] == "placebo"].copy()
        placebo = placebo[
            placebo["baseline_correct"].notna() & placebo["memory_correct"].notna()
        ]
        if not placebo.empty:
            placebo_result = add_mcnemar(paired_bootstrap_difference(
                placebo["baseline_correct"].astype(float),
                placebo["memory_correct"].astype(float),
                n_boot=N_BOOTSTRAP_POLICY, seed=SEED + 2,
            ))
            policy_rows.append({
                "model": model, "dataset": dataset, "depth": depth, "pool": pool,
                "policy": "placebo", "threshold": None,
            } | placebo_result)

policy_df = pd.DataFrame(policy_rows)
policy_df.to_csv(OUT_DIR / "analysis" / "heldout_policies.csv", index=False)

# Como a memória altera a decisão: ajuda, atrapalha ou deixa igual.
transitions = results_df[results_df["delta"].notna()].copy()
transitions["transition"] = np.select(
    [transitions["delta"] > 0, transitions["delta"] < 0],
    ["helped", "harmed"], default="unchanged",
)
transition_df = (
    transitions.groupby(["model", "dataset", "depth", "arm", "level", "transition"])
    .size().rename("n").reset_index()
)
transition_df.to_csv(OUT_DIR / "analysis" / "decision_transitions.csv", index=False)

display(policy_df.sort_values(["model", "dataset", "depth", "pool", "policy"]))


## Figuras e leitura automática

Cada figura mostra, para um modelo/dataset/profundidade:

- curva de ganho na calibração com IC 95%;
- threshold aprendido sem olhar o teste;
- efeitos no teste de sempre usar top-1, usar a política de threshold e usar
  uma memória placebo.

As tabelas CSV preservam todos os números usados nas figuras.


In [ ]:
POOL_COLORS = {"all": "tab:blue", "errors": "tab:red", "correct": "tab:green"}


def plot_result(model, dataset, depth):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    ax = axes[0]
    ax.axhline(0, color="black", linewidth=1)
    for pool in SOURCE_POOLS:
        curve = curves_df[
            (curves_df["model"] == model) & (curves_df["dataset"] == dataset)
            & (curves_df["depth"] == depth) & (curves_df["pool"] == pool)
        ]
        if curve.empty:
            continue
        color = POOL_COLORS[pool]
        ax.plot(curve["similarity"], curve["effect"], color=color, label=pool)
        if pool == "all":
            ax.fill_between(
                curve["similarity"].to_numpy(), curve["ci_low"].to_numpy(),
                curve["ci_high"].to_numpy(), color=color, alpha=0.18,
            )
        row = threshold_df[
            (threshold_df["model"] == model) & (threshold_df["dataset"] == dataset)
            & (threshold_df["depth"] == depth) & (threshold_df["pool"] == pool)
        ]
        if not row.empty and pd.notna(row.iloc[0]["threshold"]):
            ax.axvline(row.iloc[0]["threshold"], color=color, linestyle="--", alpha=0.7)
    ax.set_xlabel("similaridade da memória")
    ax.set_ylabel("ganho pareado de acurácia")
    ax.set_title("Calibração: efeito vs. similaridade")
    ax.legend(title="fonte")

    ax2 = axes[1]
    heldout = policy_df[
        (policy_df["model"] == model) & (policy_df["dataset"] == dataset)
        & (policy_df["depth"] == depth) & (policy_df["pool"] == "all")
    ].copy()
    order = ["always_top", "threshold_top", "placebo"]
    heldout["order"] = heldout["policy"].map({x: i for i, x in enumerate(order)})
    heldout = heldout.sort_values("order")
    if not heldout.empty:
        x = np.arange(len(heldout))
        y = heldout["difference"].to_numpy()
        low = y - heldout["ci_low"].to_numpy()
        high = heldout["ci_high"].to_numpy() - y
        colors = ["tab:orange" if p != "placebo" else "gray" for p in heldout["policy"]]
        ax2.bar(x, y, color=colors, alpha=0.8)
        ax2.errorbar(x, y, yerr=np.vstack([low, high]), fmt="none", color="black", capsize=4)
        ax2.set_xticks(x, heldout["policy"], rotation=20)
    ax2.axhline(0, color="black", linewidth=1)
    ax2.set_ylabel("ganho de acurácia no teste")
    ax2.set_title("Teste holdout: políticas e placebo")

    fig.suptitle(f"{model} | {dataset} | {depth} | uma memória")
    fig.tight_layout()
    path = OUT_DIR / "plots" / f"{model}__{dataset}__{depth}.png"
    fig.savefig(path, dpi=170, bbox_inches="tight")
    plt.show()


for model in STUDENT_MODELS:
    for dataset in DATASETS:
        for depth in REFLECTION_DEPTHS:
            plot_result(model, dataset, depth)


def fmt_pp(x):
    return f"{100*x:+.1f} pp"


print("\nRESULTADOS HOLDOUT COM EVIDÊNCIA POSITIVA (IC 95% acima de zero)")
positive = policy_df[
    (policy_df["policy"] == "threshold_top") & (policy_df["ci_low"] > 0)
].sort_values("difference", ascending=False)
if positive.empty:
    print("Nenhuma política de threshold teve IC 95% inteiramente positivo.")
else:
    for row in positive.itertuples():
        print(
            f"- {row.model}/{row.dataset}/{row.depth}/{row.pool}: "
            f"{fmt_pp(row.difference)} (IC {fmt_pp(row.ci_low)} a {fmt_pp(row.ci_high)}), "
            f"threshold={row.threshold:.3f}, uso={row.memory_use_rate:.1%}"
        )

print("\nRESULTADOS HOLDOUT COM EVIDÊNCIA DE DANO (IC 95% abaixo de zero)")
negative = policy_df[
    (policy_df["policy"] == "threshold_top") & (policy_df["ci_high"] < 0)
].sort_values("difference")
if negative.empty:
    print("Nenhuma política de threshold teve IC 95% inteiramente negativo.")
else:
    for row in negative.itertuples():
        print(
            f"- {row.model}/{row.dataset}/{row.depth}/{row.pool}: "
            f"{fmt_pp(row.difference)} (IC {fmt_pp(row.ci_low)} a {fmt_pp(row.ci_high)})"
        )

print("\nARQUIVOS PRINCIPAIS")
for name in (
    "all_outcomes.csv", "thresholds_calibration.csv", "effect_curves_calibration.csv",
    "heldout_policies.csv", "decision_transitions.csv",
):
    print("-", (OUT_DIR / "analysis" / name).relative_to(ROOT))


## Como interpretar

- `threshold`: primeiro ponto de ganho estimado não negativo e sustentado na
  calibração. Não é evidência suficiente por si só.
- `confident_help_threshold`: primeiro ponto em que o limite inferior do IC
  também fica positivo; pode não existir mesmo quando há cruzamento pontual.
- `threshold_identification_rate`: estabilidade do corte nos reamostragens.
  Valores baixos indicam que não há suporte para um threshold único.
- `threshold_top` em `heldout_policies.csv`: avaliação que realmente decide se
  o corte generaliza.
- `helped` e `harmed`: número de decisões que mudaram de errada para correta e
  de correta para errada.
- `placebo`: efeito de acrescentar uma memória de baixa similaridade. Se for
  semelhante ao efeito recuperado, a interpretação como transferência de
  raciocínio fica enfraquecida.

Um resultado convincente exige simultaneamente: curva plausível, threshold
estável, ganho positivo no teste, placebo inferior e replicação em mais de um
modelo pequeno.
